In [1]:
import os
import random
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
from imblearn.metrics import specificity_score
from mambapy.mamba import Mamba, MambaConfig

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory allocated at start: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Device: cuda
GPU memory allocated at start: 0.00 GB


In [2]:
class BiMambaWrapper(nn.Module):
    """
    Vision Mamba's bidirectional idea, built on mambapy's tested, verified
    Mamba block: run it once forward, once on the reversed sequence, then
    combine. The scan/recurrence math is entirely mambapy's (Blelloch
    parallel scan, numerically verified against the official implementation,
    part of Hugging Face transformers).
    """
    def __init__(self, d_model, n_layers=2, d_state=16):
        super().__init__()
        config = MambaConfig(d_model=d_model, n_layers=n_layers, d_state=d_state)
        self.mamba_fwd = Mamba(config)
        self.mamba_bwd = Mamba(config)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        # x: (B, L, d_model)
        y_fwd = self.mamba_fwd(x)
        y_bwd = torch.flip(self.mamba_bwd(torch.flip(x, dims=[1])), dims=[1])
        return self.norm(y_fwd + y_bwd)

In [3]:
class ROIPatchEmbed(nn.Module):
    """Splits each 64^3 ROI into non-overlapping 8^3 patches -> tokens.
    6 ROIs * 8*8*8 patches = 3072 tokens per subject per modality."""
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=64):
        super().__init__()
        self.grid_size = roi_size // patch_size
        self.n_tokens = (self.grid_size ** 3) * n_rois
        self.patch_conv = nn.Conv3d(1, d_model, kernel_size=patch_size, stride=patch_size)
        self.pos_embed = nn.Parameter(torch.randn(1, self.n_tokens, d_model) * 0.02)

    def forward(self, rois):
        B, N = rois.shape[0], rois.shape[1]
        toks = [self.patch_conv(rois[:, i]).flatten(2).transpose(1, 2) for i in range(N)]
        return torch.cat(toks, dim=1) + self.pos_embed


class VisionMambaBranch(nn.Module):
    """Patch embed -> bidirectional Mamba -> mean pool."""
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=64, n_layers=2, d_state=16):
        super().__init__()
        self.patch_embed = ROIPatchEmbed(n_rois, roi_size, patch_size, d_model)
        self.bimamba = BiMambaWrapper(d_model, n_layers, d_state)

    def forward(self, rois):
        tokens = self.patch_embed(rois)
        tokens = self.bimamba(tokens)
        return tokens.mean(dim=1)


class VisionMambaModel(nn.Module):
    """Single-modality model -- use for MRI-only or PET-only."""
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=64,
                 n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, rois):
        pooled = self.branch(rois)
        return self.classifier(self.dropout(pooled))


class MultimodalVisionMambaModel(nn.Module):
    """Late fusion -- separate MRI/PET branches, concatenated before classifier."""
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=64,
                 n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.mri_branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.pet_branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model * 2, n_classes)

    def forward(self, mri_rois, pet_rois):
        fused = torch.cat([self.mri_branch(mri_rois), self.pet_branch(pet_rois)], dim=1)
        return self.classifier(self.dropout(fused))

In [4]:
COHORT_CSV    = "D:/mamba_model/thesis_cohort_final.csv"
MRI_CACHE_AUG = "D:/mamba_model/preprocessed_cache_roi64_aug"
PET_CACHE_AUG = "D:/mamba_model/preprocessed_cache_pet_aug"
CKPT_DIR      = "D:/mamba_model/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

df = pd.read_csv(COHORT_CSV)
sessions = df["mri_session"].values
labels   = df["outcome_label"].values

X_tv, X_test, y_tv, y_test = train_test_split(
    sessions, labels, test_size=0.2, random_state=42, stratify=labels
)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.25, random_state=42, stratify=y_tv
)
session_to_subject = dict(zip(df["mri_session"], df["subject_id"]))

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Train: 126 | Val: 42 | Test: 42


In [5]:
class ROIDataset(Dataset):
    """Single-modality dataset (MRI-only or PET-only)."""
    def __init__(self, sessions, labels, cache_dir, is_mri=True, is_train=False):
        self.samples = []
        self.cache_dir = cache_dir
        for session_id, label in zip(sessions, labels):
            key = session_id if is_mri else session_to_subject[session_id]
            self.samples.append((key, label, "orig"))
            if is_train:
                for seed in [1, 101, 42]:
                    self.samples.append((key, label, f"aug{seed}"))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        key, label, version = self.samples[idx]
        rois = np.load(f"{self.cache_dir}/{key}_{version}.npy").astype(np.float32)
        return torch.tensor(rois).unsqueeze(1), torch.tensor(label, dtype=torch.long)


class MultimodalROIDataset(Dataset):
    """Pairs matching MRI and PET aug files per subject/seed."""
    def __init__(self, sessions, labels, mri_cache_dir, pet_cache_dir, is_train=False):
        self.samples = []
        self.mri_cache_dir = mri_cache_dir
        self.pet_cache_dir = pet_cache_dir
        for session_id, label in zip(sessions, labels):
            subject_id = session_to_subject[session_id]
            self.samples.append((session_id, subject_id, label, "orig"))
            if is_train:
                for seed in [1, 101, 42]:
                    self.samples.append((session_id, subject_id, label, f"aug{seed}"))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        mri_key, pet_key, label, version = self.samples[idx]
        mri_rois = np.load(f"{self.mri_cache_dir}/{mri_key}_{version}.npy").astype(np.float32)
        pet_rois = np.load(f"{self.pet_cache_dir}/{pet_key}_{version}.npy").astype(np.float32)
        return (torch.tensor(mri_rois).unsqueeze(1), torch.tensor(pet_rois).unsqueeze(1),
                torch.tensor(label, dtype=torch.long))

In [6]:
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

train_dataset = ROIDataset(X_train, y_train, MRI_CACHE_AUG, is_mri=True, is_train=True)
val_dataset   = ROIDataset(X_val,   y_val,   MRI_CACHE_AUG, is_mri=True, is_train=False)
test_dataset  = ROIDataset(X_test,  y_test,  MRI_CACHE_AUG, is_mri=True, is_train=False)

BATCH_SIZE = 4
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train batches/epoch: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

Train batches/epoch: 126 | Val: 11 | Test: 11


In [7]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for rois, labels in loader:
        rois, labels = rois.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(rois)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for rois, labels in loader:
            rois, labels = rois.to(device), labels.to(device)
            outputs = model(rois)
            total_loss += criterion(outputs, labels).item()
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    tpr = recall_score(all_labels, all_preds, zero_division=0)
    tnr = specificity_score(all_labels, all_preds)
    return avg_loss, acc, tpr, tnr

In [8]:
model_check = VisionMambaModel(d_model=64, n_layers=2, dropout=0.4).to(device)
opt_check = torch.optim.AdamW(model_check.parameters(), lr=1e-4)
crit_check = nn.CrossEntropyLoss(label_smoothing=0.05)

all_clean = True
for step in range(20):
    rois, labels = next(iter(train_loader))
    rois, labels = rois.to(device), labels.to(device)
    opt_check.zero_grad()
    out = model_check(rois)
    loss = crit_check(out, labels)
    loss.backward()
    grad_bad = any((torch.isnan(p.grad).any() or torch.isinf(p.grad).any())
                    for p in model_check.parameters() if p.grad is not None)
    opt_check.step()
    if grad_bad:
        all_clean = False
        print(f"step {step}: BAD GRADIENT DETECTED")
        break
    print(f"step {step}: loss={loss.item():.4f} OK")

print(f"\nAll clean: {all_clean}")

step 0: loss=0.8443 OK
step 1: loss=0.7283 OK
step 2: loss=0.7479 OK
step 3: loss=0.7234 OK
step 4: loss=0.7019 OK
step 5: loss=0.7688 OK
step 6: loss=0.6270 OK
step 7: loss=0.7368 OK
step 8: loss=0.5871 OK
step 9: loss=0.6130 OK
step 10: loss=0.6913 OK
step 11: loss=0.6816 OK
step 12: loss=0.6257 OK
step 13: loss=0.7487 OK
step 14: loss=0.6646 OK
step 15: loss=0.7342 OK
step 16: loss=0.7622 OK
step 17: loss=0.6108 OK
step 18: loss=0.5935 OK
step 19: loss=0.6329 OK

All clean: True


In [9]:
model = VisionMambaModel(d_model=64, n_layers=2, n_classes=2, dropout=0.4).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

model.train()
rois, labels = next(iter(train_loader))
rois, labels = rois.to(device), labels.to(device)

torch.cuda.synchronize()
t0 = time.time()
optimizer.zero_grad()
outputs = model(rois)
loss = criterion(outputs, labels)
loss.backward()
optimizer.step()
torch.cuda.synchronize()
first_batch_time = time.time() - t0

est_epoch_min = first_batch_time * len(train_loader) / 60
print(f"First batch: {first_batch_time:.2f}s -> estimated ~{est_epoch_min:.1f} min/epoch")

First batch: 0.38s -> estimated ~0.8 min/epoch


In [10]:
best_val_loss = float("inf")
no_improvement = 0
best_epoch = 0
save_path = f"{CKPT_DIR}/vim_mri_only_seed{SEED}.pt"

print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>10} | {'Val Acc':>8} | {'Val TPR':>8} | {'Val TNR':>8} | {'Time':>6}")
print("-" * 75)

for epoch in range(1, 101):
    t0 = time.time()
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc, val_tpr, val_tnr = evaluate(model, val_loader, criterion, device)
    scheduler.step(val_loss)
    epoch_time = time.time() - t0

    print(f"{epoch:>6} | {train_loss:>10.4f} | {val_loss:>10.4f} | "
          f"{val_acc:>8.4f} | {val_tpr:>8.4f} | {val_tnr:>8.4f} | {epoch_time:>5.1f}s")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        no_improvement = 0
        torch.save(model.state_dict(), save_path)
    else:
        no_improvement += 1
        if no_improvement >= 15:
            print(f"Early stopping at epoch {epoch}. Best: {best_epoch}")
            break

model.load_state_dict(torch.load(save_path, weights_only=True))
test_loss, test_acc, test_tpr, test_tnr = evaluate(model, test_loader, criterion, device)
print(f"\nTEST: Acc={test_acc*100:.1f}% | TPR={test_tpr*100:.1f}% | TNR={test_tnr*100:.1f}%")

 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
---------------------------------------------------------------------------
     1 |     0.7034 |     0.6943 |   0.5000 |   0.0000 |   1.0000 |  17.0s
     2 |     0.6938 |     0.6928 |   0.5000 |   0.0000 |   1.0000 |  16.7s
     3 |     0.6930 |     0.7036 |   0.5000 |   1.0000 |   0.0000 |  16.7s
     4 |     0.6918 |     0.6932 |   0.5000 |   1.0000 |   0.0000 |  17.0s
     5 |     0.6991 |     0.6899 |   0.5000 |   1.0000 |   0.0000 |  17.4s
     6 |     0.6878 |     0.6920 |   0.5000 |   1.0000 |   0.0000 |  16.8s
     7 |     0.6889 |     0.6852 |   0.5952 |   0.2381 |   0.9524 |  16.8s
     8 |     0.6801 |     0.7281 |   0.5000 |   1.0000 |   0.0000 |  16.9s
     9 |     0.6851 |     0.6847 |   0.5000 |   0.0476 |   0.9524 |  17.5s
    10 |     0.6809 |     0.6799 |   0.5476 |   1.0000 |   0.0952 |  16.9s
    11 |     0.6743 |     0.6747 |   0.5714 |   0.7143 |   0.4286 |  17.0s
    12 |     0.6670 |   

In [11]:
SEED_PET = 42
torch.manual_seed(SEED_PET)
torch.cuda.manual_seed(SEED_PET)
np.random.seed(SEED_PET)
random.seed(SEED_PET)

pet_train_dataset = ROIDataset(X_train, y_train, PET_CACHE_AUG, is_mri=False, is_train=True)
pet_val_dataset   = ROIDataset(X_val,   y_val,   PET_CACHE_AUG, is_mri=False, is_train=False)
pet_test_dataset  = ROIDataset(X_test,  y_test,  PET_CACHE_AUG, is_mri=False, is_train=False)

pet_train_loader = DataLoader(pet_train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
pet_val_loader   = DataLoader(pet_val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
pet_test_loader  = DataLoader(pet_test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"PET train batches/epoch: {len(pet_train_loader)} | Val: {len(pet_val_loader)} | Test: {len(pet_test_loader)}")

PET train batches/epoch: 126 | Val: 11 | Test: 11


In [12]:
pet_model = VisionMambaModel(d_model=64, n_layers=2, n_classes=2, dropout=0.4).to(device)
pet_criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
pet_optimizer = optim.AdamW(pet_model.parameters(), lr=1e-4, weight_decay=1e-3)
pet_scheduler = optim.lr_scheduler.ReduceLROnPlateau(pet_optimizer, mode='min', factor=0.5, patience=10)

best_val_loss = float("inf")
no_improvement = 0
best_epoch = 0
save_path = f"{CKPT_DIR}/vim_pet_only_seed{SEED_PET}.pt"

print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>10} | {'Val Acc':>8} | {'Val TPR':>8} | {'Val TNR':>8} | {'Time':>6}")
print("-" * 75)

for epoch in range(1, 101):
    t0 = time.time()
    train_loss = train_epoch(pet_model, pet_train_loader, pet_optimizer, pet_criterion, device)
    val_loss, val_acc, val_tpr, val_tnr = evaluate(pet_model, pet_val_loader, pet_criterion, device)
    pet_scheduler.step(val_loss)
    epoch_time = time.time() - t0

    print(f"{epoch:>6} | {train_loss:>10.4f} | {val_loss:>10.4f} | "
          f"{val_acc:>8.4f} | {val_tpr:>8.4f} | {val_tnr:>8.4f} | {epoch_time:>5.1f}s")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        no_improvement = 0
        torch.save(pet_model.state_dict(), save_path)
    else:
        no_improvement += 1
        if no_improvement >= 15:
            print(f"Early stopping at epoch {epoch}. Best: {best_epoch}")
            break

pet_model.load_state_dict(torch.load(save_path, weights_only=True))
test_loss, test_acc, test_tpr, test_tnr = evaluate(pet_model, pet_test_loader, pet_criterion, device)
print(f"\nPET-ONLY TEST: Acc={test_acc*100:.1f}% | TPR={test_tpr*100:.1f}% | TNR={test_tnr*100:.1f}%")

 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
---------------------------------------------------------------------------
     1 |     0.7094 |     0.7029 |   0.5000 |   1.0000 |   0.0000 |  79.6s
     2 |     0.7047 |     0.6900 |   0.5952 |   0.4286 |   0.7619 |  16.8s
     3 |     0.7052 |     0.6888 |   0.5476 |   0.1905 |   0.9048 |  16.7s
     4 |     0.6815 |     0.6987 |   0.5000 |   0.0000 |   1.0000 |  16.7s
     5 |     0.6961 |     0.6872 |   0.5238 |   0.0952 |   0.9524 |  16.7s
     6 |     0.6873 |     0.6839 |   0.5952 |   0.3810 |   0.8095 |  16.7s
     7 |     0.6821 |     0.6920 |   0.5238 |   0.0476 |   1.0000 |  16.7s
     8 |     0.6846 |     0.6824 |   0.5000 |   1.0000 |   0.0000 |  16.7s
     9 |     0.6726 |     0.6780 |   0.6190 |   0.4286 |   0.8095 |  16.7s
    10 |     0.6633 |     0.6783 |   0.5238 |   0.1429 |   0.9048 |  16.8s
    11 |     0.6551 |     0.6750 |   0.5238 |   0.9524 |   0.0952 |  16.9s
    12 |     0.6465 |   

In [13]:
SEED_MM = 42
torch.manual_seed(SEED_MM)
torch.cuda.manual_seed(SEED_MM)
np.random.seed(SEED_MM)
random.seed(SEED_MM)

mm_train_dataset = MultimodalROIDataset(X_train, y_train, MRI_CACHE_AUG, PET_CACHE_AUG, is_train=True)
mm_val_dataset   = MultimodalROIDataset(X_val,   y_val,   MRI_CACHE_AUG, PET_CACHE_AUG, is_train=False)
mm_test_dataset  = MultimodalROIDataset(X_test,  y_test,  MRI_CACHE_AUG, PET_CACHE_AUG, is_train=False)

# batch_size=4 already verified safe for multimodal (6.73GB peak measured earlier)
mm_train_loader = DataLoader(mm_train_dataset, batch_size=4, shuffle=True,  num_workers=0)
mm_val_loader   = DataLoader(mm_val_dataset,   batch_size=4, shuffle=False, num_workers=0)
mm_test_loader  = DataLoader(mm_test_dataset,  batch_size=4, shuffle=False, num_workers=0)

print(f"Multimodal train batches/epoch: {len(mm_train_loader)} | Val: {len(mm_val_loader)} | Test: {len(mm_test_loader)}")

Multimodal train batches/epoch: 126 | Val: 11 | Test: 11


In [14]:
def train_epoch_mm(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for mri_rois, pet_rois, labels in loader:
        mri_rois, pet_rois, labels = mri_rois.to(device), pet_rois.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(mri_rois, pet_rois)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate_mm(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for mri_rois, pet_rois, labels in loader:
            mri_rois, pet_rois, labels = mri_rois.to(device), pet_rois.to(device), labels.to(device)
            outputs = model(mri_rois, pet_rois)
            total_loss += criterion(outputs, labels).item()
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    tpr = recall_score(all_labels, all_preds, zero_division=0)
    tnr = specificity_score(all_labels, all_preds)
    return avg_loss, acc, tpr, tnr

In [15]:
mm_check = MultimodalVisionMambaModel(d_model=64, n_layers=2, dropout=0.4).to(device)
opt_mm_check = torch.optim.AdamW(mm_check.parameters(), lr=1e-4)
crit_mm_check = nn.CrossEntropyLoss(label_smoothing=0.05)

all_clean = True
for step in range(10):
    mri_rois, pet_rois, labels = next(iter(mm_train_loader))
    mri_rois, pet_rois, labels = mri_rois.to(device), pet_rois.to(device), labels.to(device)
    opt_mm_check.zero_grad()
    out = mm_check(mri_rois, pet_rois)
    loss = crit_mm_check(out, labels)
    loss.backward()
    grad_bad = any((torch.isnan(p.grad).any() or torch.isinf(p.grad).any())
                    for p in mm_check.parameters() if p.grad is not None)
    opt_mm_check.step()
    if grad_bad:
        all_clean = False
        print(f"step {step}: BAD GRADIENT DETECTED")
        break
    print(f"step {step}: loss={loss.item():.4f} OK")

print(f"\nAll clean: {all_clean}")

step 0: loss=0.7256 OK
step 1: loss=0.7460 OK
step 2: loss=0.6732 OK
step 3: loss=0.7396 OK
step 4: loss=0.7899 OK
step 5: loss=0.7592 OK
step 6: loss=0.6064 OK
step 7: loss=0.9843 OK
step 8: loss=0.7045 OK
step 9: loss=0.8436 OK

All clean: True


In [16]:
mm_model = MultimodalVisionMambaModel(d_model=64, n_layers=2, n_classes=2, dropout=0.4).to(device)
mm_criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
mm_optimizer = optim.AdamW(mm_model.parameters(), lr=1e-4, weight_decay=1e-3)
mm_scheduler = optim.lr_scheduler.ReduceLROnPlateau(mm_optimizer, mode='min', factor=0.5, patience=10)

best_val_loss = float("inf")
no_improvement = 0
best_epoch = 0
save_path = f"{CKPT_DIR}/vim_multimodal_seed{SEED_MM}.pt"

print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>10} | {'Val Acc':>8} | {'Val TPR':>8} | {'Val TNR':>8} | {'Time':>6}")
print("-" * 75)

for epoch in range(1, 101):
    t0 = time.time()
    train_loss = train_epoch_mm(mm_model, mm_train_loader, mm_optimizer, mm_criterion, device)
    val_loss, val_acc, val_tpr, val_tnr = evaluate_mm(mm_model, mm_val_loader, mm_criterion, device)
    mm_scheduler.step(val_loss)
    epoch_time = time.time() - t0

    print(f"{epoch:>6} | {train_loss:>10.4f} | {val_loss:>10.4f} | "
          f"{val_acc:>8.4f} | {val_tpr:>8.4f} | {val_tnr:>8.4f} | {epoch_time:>5.1f}s")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        no_improvement = 0
        torch.save(mm_model.state_dict(), save_path)
    else:
        no_improvement += 1
        if no_improvement >= 15:
            print(f"Early stopping at epoch {epoch}. Best: {best_epoch}")
            break

mm_model.load_state_dict(torch.load(save_path, weights_only=True))
test_loss, test_acc, test_tpr, test_tnr = evaluate_mm(mm_model, mm_test_loader, mm_criterion, device)
print(f"\nMULTIMODAL TEST: Acc={test_acc*100:.1f}% | TPR={test_tpr*100:.1f}% | TNR={test_tnr*100:.1f}%")

 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
---------------------------------------------------------------------------
     1 |     0.7047 |     0.7067 |   0.5000 |   0.0000 |   1.0000 |  33.8s
     2 |     0.7087 |     0.7004 |   0.5000 |   1.0000 |   0.0000 |  33.6s
     3 |     0.6970 |     0.6902 |   0.5238 |   0.0952 |   0.9524 |  33.6s
     4 |     0.6981 |     0.6843 |   0.5476 |   0.8095 |   0.2857 |  33.6s
     5 |     0.6918 |     0.7215 |   0.5000 |   1.0000 |   0.0000 |  33.7s
     6 |     0.6789 |     0.6827 |   0.5238 |   0.0952 |   0.9524 |  33.6s
     7 |     0.6694 |     0.6823 |   0.5000 |   1.0000 |   0.0000 |  33.7s
     8 |     0.6785 |     0.7032 |   0.5000 |   1.0000 |   0.0000 |  33.6s
     9 |     0.6691 |     0.6777 |   0.5238 |   1.0000 |   0.0476 |  33.7s
    10 |     0.6589 |     0.6744 |   0.5476 |   0.9524 |   0.1429 |  33.4s
    11 |     0.6554 |     0.6705 |   0.5476 |   0.9048 |   0.1905 |  33.4s
    12 |     0.6484 |   